Import

In [1]:
import sys
if 'utils' in sys.modules:
    del sys.modules['utils']
sys.path.append('../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, HeteroConv
from sklearn.metrics import roc_auc_score
import numpy as np
import os
from utils import EDGE_FEATURES

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

CONFIGS = {
    'A': ['comm'],
    'B': ['comm', 'context'],
    'C': ['comm', 'knowledge'],
    'D': ['comm', 'context', 'knowledge']
}

Device: mps


Definizione HeteroGNN

In [2]:
class HeteroGNN(nn.Module):
    def __init__(self, edge_types, hidden_channels=64):
        super().__init__()
        self.edge_types = edge_types
        
        self.conv1 = HeteroConv({
            ('node', etype, 'node'): GCNConv(-1, hidden_channels)
            for etype in edge_types
        }, aggr='sum')
        
        self.conv2 = HeteroConv({
            ('node', etype, 'node'): GCNConv(hidden_channels, hidden_channels)
            for etype in edge_types
        }, aggr='sum')
        
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        
        self.edge_classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Linear(hidden_channels, 2)
        )
    
    def forward(self, data):
        x_dict = {'node': data['node'].x}
        edge_index_dict = {
            ('node', etype, 'node'): data['node', etype, 'node'].edge_index
            for etype in self.edge_types
        }
        
        out1 = self.conv1(x_dict, edge_index_dict)
        node_emb = torch.zeros(data['node'].num_nodes, 64, device=data['node'].x.device)
        for v in out1.values():
            node_emb = node_emb + v
        node_emb = self.bn1(F.relu(node_emb))
        
        x_dict2 = {'node': node_emb}
        out2 = self.conv2(x_dict2, edge_index_dict)
        node_emb2 = torch.zeros_like(node_emb)
        for v in out2.values():
            node_emb2 = node_emb2 + v
        node_emb2 = self.bn2(F.relu(node_emb2))
        
        return node_emb2
    
    def classify_edges(self, node_emb, edge_index):
        src = node_emb[edge_index[0]]
        dst = node_emb[edge_index[1]]
        edge_emb = torch.cat([src, dst], dim=1)
        return self.edge_classifier(edge_emb)

print("HeteroGNN definita.")

HeteroGNN definita.


Funzioni di Load e training

In [3]:
def load_graphs(config_name, split):
    split_dir = f'../outputs/graphs/CONFIG_{config_name}/{split}'
    graphs = []
    for fname in sorted(os.listdir(split_dir)):
        if fname.endswith('.pt'):
            g = torch.load(f'{split_dir}/{fname}', weights_only=False)
            graphs.append(g)
    return graphs


def train_one_epoch(model, graphs, optimizer, edge_types, device):
    model.train()
    total_loss = 0
    
    for data in graphs:
        data = data.to(device)
        optimizer.zero_grad()
        
        node_emb = model(data)
        losses = []
        
        for etype in edge_types:
            edge_index = data['node', etype, 'node'].edge_index
            edge_label = data['node', etype, 'node'].edge_label
            logits = model.classify_edges(node_emb, edge_index)
            
            ce_loss = F.cross_entropy(logits, edge_label, reduction='none')
            pt = torch.exp(-ce_loss)
            focal = ((1 - pt) ** 2 * ce_loss).mean()
            losses.append(focal)
        
        loss = sum(losses)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    return total_loss / len(graphs)


def evaluate(model, graphs, edge_types, device, threshold=0.5):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    
    with torch.no_grad():
        for data in graphs:
            data = data.to(device)
            node_emb = model(data)
            
            for etype in edge_types:
                edge_index = data['node', etype, 'node'].edge_index
                edge_label = data['node', etype, 'node'].edge_label
                logits = model.classify_edges(node_emb, edge_index)
                probs = F.softmax(logits, dim=1)[:, 1]
                preds = (probs > threshold).long()
                
                all_preds.append(preds.cpu())
                all_labels.append(edge_label.cpu())
                all_probs.append(probs.cpu())
    
    all_preds  = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)
    all_probs  = torch.cat(all_probs).numpy()
    
    tp = ((all_preds == 1) & (all_labels == 1)).sum().item()
    fp = ((all_preds == 1) & (all_labels == 0)).sum().item()
    tn = ((all_preds == 0) & (all_labels == 0)).sum().item()
    fn = ((all_preds == 0) & (all_labels == 1)).sum().item()
    
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    accuracy  = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    auc       = roc_auc_score(all_labels.numpy(), all_probs)
    
    return {
        'precision': precision,
        'recall':    recall,
        'f1':        f1,
        'accuracy':  accuracy,
        'auc':       auc
    }

print("Funzioni definite.")

Funzioni definite.


Training con 5 epoch

In [4]:
EPOCHS = 5
HIDDEN = 64
results = {}

for config_name, edge_types in CONFIGS.items():
    print(f"\n{'='*40}")
    print(f"Training CONFIG_{config_name}")
    print('='*40)
    
    train_graphs = load_graphs(config_name, 'train')
    test_graphs  = load_graphs(config_name, 'test')
    
    # Filtra grafi senza anomalie dal train
    train_graphs = [g for g in train_graphs
                   if g['node', 'comm', 'node'].edge_label.sum().item() > 0]
    print(f"  Grafi train con anomalie: {len(train_graphs)}")
    
    model = HeteroGNN(edge_types=edge_types, hidden_channels=HIDDEN).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
    
    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_graphs, optimizer, edge_types, device)
        print(f"  Epoch {epoch:02d} — Loss: {loss:.4f}")
    
    metrics = evaluate(model, test_graphs, edge_types, device)
    results[config_name] = metrics
    
    print(f"\n  AUC:       {metrics['auc']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")

print("\nDone.")


Training CONFIG_A
  Grafi train con anomalie: 16
  Epoch 01 — Loss: 0.4971
  Epoch 02 — Loss: 0.1387
  Epoch 03 — Loss: 0.0817
  Epoch 04 — Loss: 0.0713
  Epoch 05 — Loss: 0.0613

  AUC:       0.9997
  Precision: 0.9842
  Recall:    0.9964
  F1:        0.9903
  Accuracy:  0.9886

Training CONFIG_B
  Grafi train con anomalie: 16
  Epoch 01 — Loss: 0.5808
  Epoch 02 — Loss: 0.2149
  Epoch 03 — Loss: 0.1600
  Epoch 04 — Loss: 0.1228
  Epoch 05 — Loss: 0.1194

  AUC:       0.9520
  Precision: 0.9392
  Recall:    0.5951
  F1:        0.7286
  Accuracy:  0.7430

Training CONFIG_C
  Grafi train con anomalie: 16
  Epoch 01 — Loss: 0.3123
  Epoch 02 — Loss: 0.1615
  Epoch 03 — Loss: 0.1058
  Epoch 04 — Loss: 0.0939
  Epoch 05 — Loss: 0.0868

  AUC:       0.8037
  Precision: 0.1005
  Recall:    0.0103
  F1:        0.0188
  Accuracy:  0.3729

Training CONFIG_D
  Grafi train con anomalie: 16
  Epoch 01 — Loss: 1.1907
  Epoch 02 — Loss: 0.2755
  Epoch 03 — Loss: 0.3113
  Epoch 04 — Loss: 0.3331
  E

In [5]:
# Salva risultati attuali
import json
with open('../outputs/results_5ep.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Risultati salvati.")

Risultati salvati.


Train con 10 epoche

In [8]:
EPOCHS = 10
HIDDEN = 64
results = {}

for config_name, edge_types in CONFIGS.items():
    print(f"\n{'='*40}")
    print(f"Training CONFIG_{config_name}")
    print('='*40)
    
    train_graphs = load_graphs(config_name, 'train')
    test_graphs  = load_graphs(config_name, 'test')
    
    # Filtra grafi senza anomalie dal train
    train_graphs = [g for g in train_graphs
                   if g['node', 'comm', 'node'].edge_label.sum().item() > 0]
    print(f"  Grafi train con anomalie: {len(train_graphs)}")
    
    model = HeteroGNN(edge_types=edge_types, hidden_channels=HIDDEN).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
    
    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_graphs, optimizer, edge_types, device)
        print(f"  Epoch {epoch:02d} — Loss: {loss:.4f}")
    
    metrics = evaluate(model, test_graphs, edge_types, device)
    results[config_name] = metrics
    
    print(f"\n  AUC:       {metrics['auc']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1:        {metrics['f1']:.4f}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")

print("\nDone.")


Training CONFIG_A
  Grafi train con anomalie: 16
  Epoch 01 — Loss: 0.3802
  Epoch 02 — Loss: 0.1281
  Epoch 03 — Loss: 0.1110
  Epoch 04 — Loss: 0.1124
  Epoch 05 — Loss: 0.1345
  Epoch 06 — Loss: 0.1494
  Epoch 07 — Loss: 0.1556
  Epoch 08 — Loss: 0.1315
  Epoch 09 — Loss: 0.1131
  Epoch 10 — Loss: 0.0964

  AUC:       0.9989
  Precision: 0.9961
  Recall:    0.9859
  F1:        0.9910
  Accuracy:  0.9896

Training CONFIG_B
  Grafi train con anomalie: 16
  Epoch 01 — Loss: 0.4553
  Epoch 02 — Loss: 0.1687
  Epoch 03 — Loss: 0.1664
  Epoch 04 — Loss: 0.1494
  Epoch 05 — Loss: 0.1375
  Epoch 06 — Loss: 0.1221
  Epoch 07 — Loss: 0.1096
  Epoch 08 — Loss: 0.0983
  Epoch 09 — Loss: 0.0838
  Epoch 10 — Loss: 0.0807

  AUC:       0.9860
  Precision: 0.9791
  Recall:    0.4007
  F1:        0.5687
  Accuracy:  0.6478

Training CONFIG_C
  Grafi train con anomalie: 16
  Epoch 01 — Loss: 0.4697
  Epoch 02 — Loss: 0.1959
  Epoch 03 — Loss: 0.1743
  Epoch 04 — Loss: 0.1380
  Epoch 05 — Loss: 0.125

In [7]:
import json

# Converti risultati in formato serializzabile
results_serializable = {
    config: {k: float(v) for k, v in metrics.items()}
    for config, metrics in results.items()
}

with open('../outputs/results_5ep.json', 'w') as f:
    json.dump(results_serializable, f, indent=2)
print("Risultati salvati.")

Risultati salvati.


In [9]:
import json

results_10ep = {
    'A': {'auc': 0.9989, 'precision': 0.9961, 'recall': 0.9859, 'f1': 0.9910, 'accuracy': 0.9896},
    'B': {'auc': 0.9860, 'precision': 0.9791, 'recall': 0.4007, 'f1': 0.5687, 'accuracy': 0.6478},
    'C': {'auc': 0.9995, 'precision': 0.9988, 'recall': 0.9824, 'f1': 0.9905, 'accuracy': 0.9891},
    'D': {'auc': 0.9645, 'precision': 0.9855, 'recall': 0.4092, 'f1': 0.5783, 'accuracy': 0.6542},
}

with open('../outputs/results_10ep.json', 'w') as f:
    json.dump(results_10ep, f, indent=2)
print("Salvato.")

Salvato.
